In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy import stats
from scipy.stats import norm
import requests
import os
import plotly.graph_objects as go
import dataframe_image as dfi
import time
from bs4 import BeautifulSoup # library to parse HTML documents
from collections import Counter
import re
import pyfolio as pf
import itertools
from itertools import chain
import matplotlib_venn
import matplotlib.ticker as mtick
from matplotlib_venn import venn3
import networkx as nx
from upsetplot import from_contents
from upsetplot import plot
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.lines as mlines
from matplotlib.dates import DateFormatter, DayLocator
import imgkit
%matplotlib inline

import cvxpy as cp
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns

import openai

# import cplex

import cvxpy as cp
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
import plotly.express as px


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(


(CVXPY) Mar 01 03:00:15 PM: Encountered unexpected exception importing solver CBC:
ImportError("dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/cylp/cy/CyCoinIndexedVector.cpython-311-darwin.so, 0x0002): symbol not found in flat namespace '__ZN9CoinError12printErrors_E'")


In [2]:
## In-sample end date (using 5 years of weekly data prior to end date)
insample_enddate = datetime(2023, 3, 31)

## Out-of-sample period (1 Apr 2023 to 1 Dec 2023) (using daily data))
outsample_startdate = datetime(2023, 4, 1)
outsample_enddate   = datetime(2023, 12,  1)

insample_startdate = insample_enddate - timedelta(weeks=5*52) # five years prior to April 2023

In [4]:
index_tickers = ["^GSPC", "^GSPE", '^SP500-15', '^SP500-20', '^SP500-25', '^SP500-30', '^SP500-35', '^SP500-40',\
                 '^SP500-45', '^SP500-50', '^SP500-55', '^SP500-60']


index_data_outs = pd.DataFrame()

# Download historical data for each index for out-of-sample period 1
for ticker in index_tickers:
    index_df = yf.download(ticker, start=outsample_startdate, end=outsample_enddate, interval='1d')
    index_df['Index_Return'] = index_df['Adj Close'].pct_change()
    index_df.dropna(subset=['Index_Return'], inplace=True)
    index_data_outs[ticker] = index_df['Index_Return']

# Set the index of out-of-sample data to datetime format
index_data_outs.index = pd.to_datetime(index_data_outs.index)

# Create empty lists to store the tickers with continuous data in each out-of-sample period
valid_index_tickers_outs = []

for ticker in index_tickers:
    # Check if the ticker has continuous data in out-of-sample period 1
    if ticker in index_data_outs.columns and index_data_outs[ticker].notna().all():
        valid_index_tickers_outs.append(ticker)


index_tickers = valid_index_tickers_outs

print("Valid Index Tickers in Out-of-Sample Period")
print(valid_index_tickers_outs)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
Valid Index Tickers in Out-of-Sample Period
['^GSPC', '^GSPE', '^SP500-15', '^SP500-20', '^SP500-25', '^SP500-30', '^SP500-35', '^SP500-40', '^SP500-45', '^SP500-50', '^SP5

In [6]:
index_data_outs

,^GSPC,^GSPE,^SP500-15,^SP500-20,^SP500-25,^SP500-30,^SP500-35,^SP500-40,^SP500-45,^SP500-50,^SP500-55,^SP500-60
Date,,,,,,,,,,,,
2023-04-04,-0.005797,-0.017205,-0.014671,-0.022524,-0.000720,-0.002435,0.000236,-0.010124,-0.005819,0.003064,0.005212,0.000859
2023-04-05,-0.002492,0.014207,-0.002223,-0.013005,-0.020381,0.005547,0.017304,-0.001413,-0.011920,-0.002175,0.025721,-0.005064
2023-04-06,0.003579,-0.014716,-0.002189,-0.000254,0.000496,0.000801,0.002252,0.003074,0.006785,0.017072,0.007355,0.006987
2023-04-10,0.000996,0.006466,0.004925,0.008976,0.004249,-0.000140,-0.000372,0.002708,-0.001472,-0.006888,-0.002002,0.005097
2023-04-11,-0.000041,0.008916,0.007282,0.005891,0.000159,0.002413,0.002961,0.008495,-0.010330,-0.004264,0.000650,0.004517
...,...,...,...,...,...,...,...,...,...,...,...,...
2023-11-24,0.000597,0.004469,0.003326,0.002915,0.001336,0.003824,0.005107,0.003298,-0.003182,-0.006658,0.003042,0.003232
2023-11-27,-0.001954,-0.003968,-0.000844,-0.005792,0.001926,-0.002396,-0.006353,-0.002810,0.000441,-0.004746,0.000948,0.004280
2023-11-28,0.000980,0.000576,0.001963,-0.002420,0.005379,0.004021,-0.005034,-0.001025,0.001912,0.003263,0.003125,0.005800


In [7]:
temp_stock_return = index_data_outs
cumulative_returns = pd.DataFrame(index=temp_stock_return.index)

In [8]:
cumulative_returns

""
Date
2023-04-04
2023-04-05
2023-04-06
2023-04-10
2023-04-11
...
2023-11-24
2023-11-27
2023-11-28


In [12]:
for ind in index_tickers:
    cumulative_returns[ind] = (1 + index_data_outs[ind]).cumprod()
    print(ind)

cumulative_returns.index = pd.to_datetime(cumulative_returns.index)
outsample_data = cumulative_returns.loc[outsample_startdate:outsample_enddate]

fig = go.Figure()

for column in outsample_data.columns:
    fig.add_trace(go.Scatter(x=outsample_data.index,
                                y=outsample_data[column],
                                mode='lines',
                                name=column))

fig.update_layout(
    title=f'Cumulative Returns of sector indexes from Apr 23 to Dec 23',
    xaxis_title='Date',
    yaxis_title='Cumulative Returns',
    hovermode="x unified"
)

fig.show()

^GSPC
^GSPE
^SP500-15
^SP500-20
^SP500-25
^SP500-30
^SP500-35
^SP500-40
^SP500-45
^SP500-50
^SP500-55
^SP500-60


In [15]:
cumulative_returns.sum().sort_values()

^SP500-55    159.537017
^SP500-30    162.289151
^SP500-60    162.970922
^GSPE        163.009667
^SP500-15    164.663162
^SP500-35    167.210030
^SP500-20    170.071948
^SP500-40    173.642073
^GSPC        175.803395
^SP500-25    183.218524
^SP500-45    186.863186
^SP500-50    189.241203
dtype: float64